In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42

In [6]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print("Train shape:", train.shape, " Test shape:", test.shape)

Train shape: (6818, 13)  Test shape: (1705, 12)


In [7]:
train['product_category'] = train['product_category'].str.strip().str.title()
test['product_category'] = test['product_category'].str.strip().str.title()

weight_by_product = train.groupby('product_code')['product_weight_kg'].mean()
weight_by_category = train.groupby('product_category')['product_weight_kg'].mean()
train['product_weight_kg'] = train['product_weight_kg'].fillna(train['product_code'].map(weight_by_product))
train['product_weight_kg'] = train['product_weight_kg'].fillna(train['product_category'].map(weight_by_category))
test['product_weight_kg'] = test['product_weight_kg'].fillna(test['product_code'].map(weight_by_product))
test['product_weight_kg'] = test['product_weight_kg'].fillna(test['product_category'].map(weight_by_category))

train['store_size'] = train['store_size'].fillna('Unknown')
test['store_size'] = test['store_size'].fillna('Unknown')

train.loc[train['shelf_visibility'] == 0, 'shelf_visibility'] = np.nan
test.loc[test['shelf_visibility'] == 0, 'shelf_visibility'] = np.nan
vis_by_category = train.groupby('product_category')['shelf_visibility'].mean()
train['shelf_visibility'] = train['shelf_visibility'].fillna(train['product_category'].map(vis_by_category))
test['shelf_visibility'] = test['shelf_visibility'].fillna(test['product_category'].map(vis_by_category))

for df in [train, test]:
    df['price_per_kg'] = df['product_price'] / df['product_weight_kg']
    df['visibility_x_price'] = df['shelf_visibility'] * df['product_price']
    df['store_age_bucket'] = pd.cut(df['store_age_years'], bins=[0, 25, 35, 45, 100],
                                     labels=['New', 'Mid', 'Mature', 'Old']).astype(str)

cat_cols = ['fat_content', 'product_category', 'store_code', 'store_size',
            'store_location_tier', 'store_format', 'store_age_bucket']

base_features = ['product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category',
                  'product_price', 'store_code', 'store_age_years', 'store_size',
                  'store_location_tier', 'store_format', 'price_per_kg', 'visibility_x_price',
                  'store_age_bucket']
te_features = ['product_avg_sales', 'product_sales_count', 'store_avg_sales', 'category_avg_sales']
feature_cols = base_features + te_features
catboost_feature_cols = feature_cols + ['product_code']  # CatBoost also gets raw product_code

GLOBAL_MEAN = train['total_sales'].mean()

In [8]:
SMOOTH_M = {'product_code': 15, 'product_category': 30, 'store_code': 50}


def smooth_encode(fit_df, key, m):
    stats = fit_df.groupby(key, observed=True)['total_sales'].agg(['mean', 'count'])
    stats['enc'] = (stats['count'] * stats['mean'] + m * GLOBAL_MEAN) / (stats['count'] + m)
    return stats['enc'], stats['count']


def add_target_encodings(fit_df, d):
    """Smoothed target encoding, fit on fit_df only, applied to d."""
    prod_enc, prod_cnt = smooth_encode(fit_df, 'product_code', SMOOTH_M['product_code'])
    cat_enc, _ = smooth_encode(fit_df, 'product_category', SMOOTH_M['product_category'])
    store_enc, _ = smooth_encode(fit_df, 'store_code', SMOOTH_M['store_code'])
    d['category_avg_sales'] = d['product_category'].map(cat_enc).fillna(GLOBAL_MEAN)
    d['store_avg_sales'] = d['store_code'].map(store_enc).fillna(GLOBAL_MEAN)
    d['product_avg_sales'] = d['product_code'].map(prod_enc).fillna(d['category_avg_sales'])
    d['product_sales_count'] = d['product_code'].map(prod_cnt).fillna(0)
    return d

In [9]:
def run_cv(kind, params, n_folds):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_rmses = []

    for tr_idx, val_idx in kf.split(train):
        fold_train = train.iloc[tr_idx].reset_index(drop=True).copy()
        fold_val = train.iloc[val_idx].reset_index(drop=True).copy()
        fold_test = test.copy()

        for d in [fold_train, fold_val, fold_test]:
            add_target_encodings(fold_train if d is not fold_train else fold_train, d)
        # (target encodings above are always fit on fold_train, applied to each d)

        cols = catboost_feature_cols if kind == 'cat' else feature_cols
        for d in [fold_train, fold_val, fold_test]:
            for c in cat_cols:
                d[c] = d[c].astype('category' if kind != 'cat' else str)

        X_tr, y_tr = fold_train[cols], fold_train['total_sales']
        X_val, y_val = fold_val[cols], fold_val['total_sales']
        X_te = fold_test[cols]

        if kind == 'lgb':
            model = lgb.LGBMRegressor(**params, random_state=RANDOM_STATE, verbosity=-1)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='rmse',
                      callbacks=[lgb.early_stopping(80, verbose=False)])
            val_pred = model.predict(X_val)
            te_pred = model.predict(X_te)

        elif kind == 'xgb':
            model = xgb.XGBRegressor(**params, random_state=RANDOM_STATE, enable_categorical=True,
                                      tree_method='hist', early_stopping_rounds=80, eval_metric='rmse')
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            val_pred = model.predict(X_val)
            te_pred = model.predict(X_te)

        elif kind == 'cat':
            cat_idx = [cols.index(c) for c in cat_cols + ['product_code']]
            train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
            val_pool = Pool(X_val, y_val, cat_features=cat_idx)
            model = CatBoostRegressor(**params, random_seed=RANDOM_STATE, loss_function='RMSE',
                                       early_stopping_rounds=80, verbose=False)
            model.fit(train_pool, eval_set=val_pool, use_best_model=True)
            val_pred = model.predict(Pool(X_val, cat_features=cat_idx))
            te_pred = model.predict(Pool(X_te, cat_features=cat_idx))

        val_pred = np.clip(val_pred, 0, None)
        oof[val_idx] = val_pred
        fold_rmses.append(np.sqrt(mean_squared_error(y_val, val_pred)))
        test_pred += np.clip(te_pred, 0, None) / n_folds

    return oof, test_pred, fold_rmses


# ---- Optuna tuning (cheap 3-fold CV as the objective, 20 trials each) ----
def tune_lgb(trial):
    params = dict(
        objective='regression', metric='rmse', n_estimators=1500,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 60),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
    )
    _, _, rmses = run_cv('lgb', params, n_folds=3)
    return np.mean(rmses)


def tune_xgb(trial):
    params = dict(
        n_estimators=1500,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 20),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
    )
    _, _, rmses = run_cv('xgb', params, n_folds=3)
    return np.mean(rmses)


def tune_cat(trial):
    params = dict(
        iterations=1500,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        depth=trial.suggest_int('depth', 4, 10),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 15),
    )
    _, _, rmses = run_cv('cat', params, n_folds=3)
    return np.mean(rmses)


print("\nTuning LightGBM...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(tune_lgb, n_trials=20)

print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(tune_xgb, n_trials=20)

print("Tuning CatBoost...")
study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(tune_cat, n_trials=20)

# final 5-fold CV with best params found
lgb_params = dict(objective='regression', metric='rmse', n_estimators=2000, **study_lgb.best_params)
xgb_params = dict(n_estimators=2000, **study_xgb.best_params)
cat_params = dict(iterations=2000, **study_cat.best_params)

lgb_oof, lgb_test, _ = run_cv('lgb', lgb_params, n_folds=5)
xgb_oof, xgb_test, _ = run_cv('xgb', xgb_params, n_folds=5)
cat_oof, cat_test, _ = run_cv('cat', cat_params, n_folds=5)


Tuning LightGBM...
Tuning XGBoost...
Tuning CatBoost...


In [10]:
blend_oof = (lgb_oof + xgb_oof + cat_oof) / 3
blend_test = (lgb_test + xgb_test + cat_test) / 3

submission = pd.DataFrame({'id': test['id'], 'total_sales': blend_test})
submission.to_csv('submission_v2.csv', index=False)


In [11]:
y_true = train['total_sales']
print(f"\nLightGBM RMSE: {np.sqrt(mean_squared_error(y_true, lgb_oof)):.2f}")
print(f"XGBoost  RMSE: {np.sqrt(mean_squared_error(y_true, xgb_oof)):.2f}")
print(f"CatBoost RMSE: {np.sqrt(mean_squared_error(y_true, cat_oof)):.2f}")
print(f"Blend    RMSE: {np.sqrt(mean_squared_error(y_true, blend_oof)):.2f}")
print("\nSaved predictions to submission_v2.csv")


LightGBM RMSE: 1151.95
XGBoost  RMSE: 1164.38
CatBoost RMSE: 1183.62
Blend    RMSE: 1162.28

Saved predictions to submission_v2.csv
